In [ ]:
## Notebook 05 — Logistic Regression PD Model (Baseline)

**Input:** ready_train/val/test_labeled.parquet — 104 features + default_flag  
**Output:** Calibrated pipeline, coefficient table, calibration table, lift table  
**Purpose:** Train, evaluate, and calibrate an interpretable logistic regression 
PD model as the baseline champion against which all challenger models are benchmarked.

---

### Why Logistic Regression First

Logistic regression is the mandatory starting point for PD modeling in regulated 
banking environments:

- **Interpretability:** every prediction decomposes into feature contributions via 
  log-odds, auditable by a credit committee or regulator
- **Regulatory acceptance:** Basel II IRB and RBI model governance frameworks favor 
  models whose outputs can be validated at the feature level
- **Calibration benchmark:** any challenger model (XGBoost V1, V2) must outperform 
  this baseline on both discrimination AND calibration to justify reduced interpretability

---

### Key Modeling Choices

| Choice | Decision | Justification |
|---|---|---|
| Scaler | RobustScaler, numeric features only | Credit data retains outliers even after winsorization in NB03. Median/IQR is less distorted by extreme income or loan values than mean/std |
| Binary features | Passthrough | OHE and flag columns are already {0,1} — scaling is unnecessary and would break interpretation |
| Class imbalance | class_weight='balanced' | Default rate is 19.97%. Without upweighting, the model is biased toward predicting the majority class |
| Regularization | L2, C=1.0 | Standard baseline. Provides implicit shrinkage of correlated coefficients. No tuning performed to preserve a clean, comparable baseline |
| Solver | lbfgs | Standard for L2 penalty at this data scale (~940k rows) |

In [ ]:
### Phase 0 — Pre-flight Quality Checks
Full data integrity validation across train/val/test before any modeling begins.
Confirms target distribution, zero NaNs, zero duplicates, and exact column alignment 
with the canonical feature list.

In [1]:
# Phase 0 QC script


import pandas as pd
import json
import os
import numpy as np

# ---------- CONFIG: filepaths  ----------
CANONICAL_JSON = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/canonical_feature_list.json"
TRAIN_PATH = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_train_labeled.parquet"
VAL_PATH   = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_val_labeled.parquet"
TEST_PATH  = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_test_labeled.parquet"

TARGET = "default_flag"

# ---------- Helpers ----------
def load_parquet(path, reset_index=True):
    print(f"\nLoading: {path}")
    df = pd.read_parquet(path)
    if reset_index:
        df = df.reset_index(drop=True)
    print(f" -> shape: {df.shape}, memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
    return df

def percent_str(n, total):
    return f"{n} ({100.0 * n / total:.4f}%)"

# ---------- 1) Load canonical features ----------
print("1) Loading canonical feature list...")
with open(CANONICAL_JSON, "r") as f:
    canonical_obj = json.load(f)

canonical_features = canonical_obj.get("features") or canonical_obj.get("feature_list") or canonical_obj.get("features", [])
ordered_columns_with_target_last = canonical_obj.get("ordered_columns_with_target_last", None)

if not canonical_features:
    raise RuntimeError("Canonical feature list appears empty; inspect JSON at: " + CANONICAL_JSON)

print(f" -> canonical features count: {len(canonical_features)}")
if ordered_columns_with_target_last:
    print(" -> ordered_columns_with_target_last available in JSON (will be used for strict order check).")

# ---------- 2) Load splits (reset index for safety) ----------
df_train = load_parquet(TRAIN_PATH, reset_index=True)
df_val   = load_parquet(VAL_PATH, reset_index=True)
df_test  = load_parquet(TEST_PATH, reset_index=True)

# ---------- 3) Basic target distribution checks ----------
print("\n2) TARGET distribution checks (counts & proportions):")
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    total = len(df)
    if TARGET not in df.columns:
        raise KeyError(f"Target column '{TARGET}' not found in {name} dataset.")
    vc = df[TARGET].value_counts(dropna=False)
    zeros = int(vc.get(0, 0))
    ones  = int(vc.get(1, 0))
    nnull = int(df[TARGET].isna().sum())
    print(f"\n - {name}: rows={total}, nulls_in_target={nnull}")
    print(f"    0 -> {percent_str(zeros, total)}")
    print(f"    1 -> {percent_str(ones, total)}")
    if nnull > 0:
        print("    !!! WARNING: nulls present in target in", name)

# ---------- 4) Duplicate ROWS detection (report only) ----------
print("\n3) Duplicate ROWS detection (no rows are dropped by this script).")
def duplicate_rows_report(df, name, sample_n=10):
    dup_mask = df.duplicated(keep=False)  # mark all duplicates so we can sample
    n_dup_rows = int(dup_mask.sum())
    print(f"\n - {name}: total duplicate-row entries = {n_dup_rows} (rows that have duplicates in the DF).")
    if n_dup_rows > 0:
        # show some groups of duplicates
        dup_df = df[dup_mask].copy()
        # show counts per duplicated signature (group by all columns)
        group_sizes = dup_df.groupby(list(dup_df.columns)).size().reset_index(name="dup_count").sort_values("dup_count", ascending=False)
        print(f"   Unique duplicate signatures (different identical-row groups): {len(group_sizes)}")
        print("   Top duplicate signature counts (show up to 10):")
        print(group_sizes.head(10).to_string(index=False))
        # show sample of actual duplicate rows (first few)
        print("\n   Sample duplicated rows (first rows of each signature):")
        # pick up to `sample_n` signature rows to preview
        for i, row in group_sizes.head(sample_n).iterrows():
            # build boolean mask for the signature
            sig_mask = (dup_df[list(canonical_features + [TARGET])].eq(row[canonical_features + [TARGET]].values).all(axis=1)) \
                        if set(canonical_features + [TARGET]).issubset(df.columns) else None
            # fallback: just show the first matching duplicates by index
            example = dup_df.iloc[:1]
            print(example.iloc[:, :10].to_string())  # print first 10 cols to keep preview compact
            break
    else:
        print("   No duplicate rows detected.")
    return n_dup_rows

dup_train = duplicate_rows_report(df_train, "train")
dup_val   = duplicate_rows_report(df_val, "val")
dup_test  = duplicate_rows_report(df_test, "test")

# Note: we only report duplicates. .

# ---------- 5) Confirm canonical feature list matches splits EXACTLY (name + order) ----------
print("\n4) Canonical feature vs DataFrame columns check (strict):")
# Build expected ordered columns: canonical_features + [TARGET]
expected_order = canonical_features + [TARGET]

def compare_columns(expected, df, name):
    cols = list(df.columns)
    equal = cols == expected
    if equal:
        print(f" - {name}: exact match with canonical ordered list.")
        return True, []
    # if not equal, print diagnostics
    print(f" - {name}: MISMATCH with canonical ordered list.")
    # find first mismatch index
    min_len = min(len(cols), len(expected))
    mismatches = []
    for i in range(min_len):
        if cols[i] != expected[i]:
            mismatches.append(("pos", i, "expected", expected[i], "found", cols[i]))
            break
    # extra columns at the end or missing
    extra = [c for c in cols if c not in expected]
    missing = [c for c in expected if c not in cols]
    print(f"   Total columns in df: {len(cols)}, expected: {len(expected)}")
    print(f"   Example mismatch (first): {mismatches[:1]}")
    print(f"   Number of extra columns in df not in canonical: {len(extra)} (show up to 10): {extra[:10]}")
    print(f"   Number of missing columns from df that are in canonical: {len(missing)} (show up to 10): {missing[:10]}")
    return False, {"mismatch_example": mismatches, "extra": extra, "missing": missing}

ok_train, train_diff = compare_columns(expected_order, df_train, "train")
ok_val,   val_diff   = compare_columns(expected_order, df_val,   "val")
ok_test,  test_diff  = compare_columns(expected_order, df_test,  "test")

# ---------- 6) Check NaNs in all columns (paranoid check) ----------
print("\n5) Full NaN check across all columns (per-split):")
def nan_summary(df, name, top_n=20):
    nulls = df.isna().sum()
    nonzero = nulls[nulls > 0].sort_values(ascending=False)
    if nonzero.empty:
        print(f" - {name}: No NaNs detected in any column.")
    else:
        print(f" - {name}: columns with NaNs (show up to {top_n}):")
        print(nonzero.head(top_n).to_string())

nan_summary(df_train, "train")
nan_summary(df_val, "val")
nan_summary(df_test, "test")

# ---------- 7) Phase 0 summary ----------
print("\n\nPHASE 0 SUMMARY")
print("----------------")
print(f" - canonical features count: {len(canonical_features)}")
print(f" - expected total columns (features + target): {len(expected_order)}")
print(f" - train shape: {df_train.shape}")
print(f" - val   shape: {df_val.shape}")
print(f" - test  shape: {df_test.shape}")
print(f" - target converted earlier? dtype(train): {df_train[TARGET].dtype}, dtype(val): {df_val[TARGET].dtype}, dtype(test): {df_test[TARGET].dtype}")
print(f" - duplicates (rows) counts: train={dup_train}, val={dup_val}, test={dup_test}")
print("\nNotes:")
print(" - Duplicate ROWS are reported above but NOT dropped.")
print(" - Canonical ordering check printed above; if any split mismatches, print details and fix before modeling.")
print(" - Index was reset on load to ensure clean 0..N-1 ordering (safe standard practice).")
print("\nPhase 0 checks complete.")


1) Loading canonical feature list...
 -> canonical features count: 104
 -> ordered_columns_with_target_last available in JSON (will be used for strict order check).

Loading: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_train_labeled.parquet
 -> shape: (941745, 105), memory: 142.80 MB

Loading: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_val_labeled.parquet
 -> shape: (201802, 105), memory: 30.60 MB

Loading: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_test_labeled.parquet
 -> shape: (201803, 105), memory: 30.60 MB

2) TARGET distribution checks (counts & proportions):

 - train: rows=941745, nulls_in_target=0
    0 -> 753726 (80.0350%)
    1 -> 188019 (19.9650%)

 - val: rows=201802, nulls_in_target=0
    0 -> 161512 (80.0349%)
    1 -> 40290 (19.9651%)

 - test: rows=201803, nulls_in_target=0
    0 -> 161513 (80.0350%)
    1 -> 40290 (19.9650%)

3) Duplicate ROWS detection (no rows are dropped by this script).

 - train: t

In [ ]:
### Phase 1 — Feature Classification
104 features split into 18 numeric (float32, require scaling) and 86 binary/OHE 
(int8, passthrough). Both lists saved as JSON for reproducible scoring.

In [2]:
# Phase 1 Step-1: detect numeric vs binary OHE features, save lists, and inspect numeric distributions
import os
import json
import numpy as np
import pandas as pd

# ---------- CONFIG ----------
ARTIFACT_DIR = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts"
NUM_JSON = os.path.join(ARTIFACT_DIR, "numeric_features.json")
BIN_JSON = os.path.join(ARTIFACT_DIR, "binary_ohe_features.json")
CANONICAL_JSON = os.path.join(ARTIFACT_DIR, "canonical_feature_list.json")
GUIDE_PATH = "/mnt/data/Credit_Risk_Project_Guide.pdf"   # uploaded guide (informational only)
TARGET = "default_flag"
TRAIN_DF = globals().get("df_train")
VAL_DF   = globals().get("df_val")
TEST_DF  = globals().get("df_test")

# ---------- basic checks ----------
if TRAIN_DF is None or VAL_DF is None or TEST_DF is None:
    raise RuntimeError("One or more of df_train, df_val, df_test not found in memory. Load them first.")

print("Shapes: train", TRAIN_DF.shape, "val", VAL_DF.shape, "test", TEST_DF.shape)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------- 1) detect numeric and binary lists ----------
# Prefer float32; if none found, fall back to any float dtype.
numeric_features = [c for c in TRAIN_DF.columns if str(TRAIN_DF[c].dtype) == "float32"]
if len(numeric_features) == 0:
    numeric_features = [c for c in TRAIN_DF.columns if str(TRAIN_DF[c].dtype).startswith("float")]

binary_ohe_features = [c for c in TRAIN_DF.columns if str(TRAIN_DF[c].dtype).startswith("int8") and c != TARGET]

print(f"\nDetected numeric features (count={len(numeric_features)}):")
print(numeric_features)
print(f"\nDetected binary/OHE features (count={len(binary_ohe_features)}). Showing first 60:")
print(binary_ohe_features[:60])

# ---------- 2) save lists ----------
with open(NUM_JSON, "w") as f:
    json.dump(numeric_features, f, indent=2)
with open(BIN_JSON, "w") as f:
    json.dump(binary_ohe_features, f, indent=2)

print(f"\nSaved numeric feature list to: {NUM_JSON}")
print(f"Saved binary/OHE feature list to: {BIN_JSON}")
print(f"Canonical feature JSON used for ordering (if needed): {CANONICAL_JSON}")
print(f"Uploaded project guide (informational): {GUIDE_PATH}")

# ---------- 3) numeric diagnostics on train (and quick val/test checks) ----------
if len(numeric_features) == 0:
    print("\nNo numeric features detected. Nothing more to inspect.")
else:
    diag_rows = []
    for col in numeric_features:
        s_train = TRAIN_DF[col].dropna().astype("float64")
        s_val = VAL_DF[col].dropna().astype("float64") if col in VAL_DF.columns else pd.Series(dtype="float64")
        s_test = TEST_DF[col].dropna().astype("float64") if col in TEST_DF.columns else pd.Series(dtype="float64")

        p01 = float(np.nanpercentile(s_train, 1)) if len(s_train) else np.nan
        p50 = float(np.nanpercentile(s_train, 50)) if len(s_train) else np.nan
        p99 = float(np.nanpercentile(s_train, 99)) if len(s_train) else np.nan
        mean = float(s_train.mean()) if len(s_train) else np.nan
        sd = float(s_train.std(ddof=0)) if len(s_train) else np.nan
        skew = float(s_train.skew()) if len(s_train) else np.nan

        pct_below_p01 = 100.0 * (s_train < p01).sum() / len(s_train) if len(s_train) else 0.0
        pct_above_p99 = 100.0 * (s_train > p99).sum() / len(s_train) if len(s_train) else 0.0
        pct_outside_3sd = 100.0 * ((s_train < (mean - 3*sd)) | (s_train > (mean + 3*sd))).sum() / len(s_train) if len(s_train) else 0.0

        # quick relative checks for val/test: median differences relative to train sd
        val_med_diff = (s_val.median() - p50) / sd if (len(s_val) and sd != 0) else np.nan
        test_med_diff = (s_test.median() - p50) / sd if (len(s_test) and sd != 0) else np.nan

        diag_rows.append({
            "feature": col,
            "count_train": int(len(s_train)),
            "mean_train": mean,
            "sd_train": sd,
            "p01_train": p01,
            "p50_train": p50,
            "p99_train": p99,
            "skew_train": skew,
            "pct_below_p01": round(pct_below_p01, 4),
            "pct_above_p99": round(pct_above_p99, 4),
            "pct_outside_mean_3sd": round(pct_outside_3sd, 4),
            "val_median_minus_train_med_over_sd": None if np.isnan(val_med_diff) else round(val_med_diff, 4),
            "test_median_minus_train_med_over_sd": None if np.isnan(test_med_diff) else round(test_med_diff, 4)
        })

    diag_df = pd.DataFrame(diag_rows).set_index("feature")
    pd.options.display.float_format = '{:,.6f}'.format
    print("\nNumeric diagnostics (train) — sorted by pct_outside_mean_3sd desc:")
    print(diag_df.sort_values("pct_outside_mean_3sd", ascending=False).to_string(max_rows=200))

    # heuristic to recommend scaler
    num_outlier_features = (diag_df["pct_outside_mean_3sd"] > 1.0).sum()
    num_heavy_tail_features = (diag_df["pct_above_p99"] > 0.5).sum()

    print("\nHeuristic summary:")
    print(f" - Numeric features with >1% outside mean±3sd: {num_outlier_features}")
    print(f" - Numeric features with >0.5% above 99th percentile: {num_heavy_tail_features}")

    if (num_outlier_features >= 1) or (num_heavy_tail_features >= 3):
        suggested = "RobustScaler (median/IQR) — recommended due to heavy tails/outliers"
    else:
        suggested = "StandardScaler (mean/std) — recommended (no extreme tails found)"

    print(f"\nSuggested scaler (heuristic): {suggested}")
    print("\nWhen ready, reply with: A (StandardScaler), B (RobustScaler), or C (auto-pick & build pipeline).")


Shapes: train (941745, 105) val (201802, 105) test (201803, 105)

Detected numeric features (count=18):
['annual_inc', 'credit_history_years', 'delinq_2yrs', 'dti', 'emp_length_yrs', 'fico_mean', 'funded_amnt', 'grade', 'inq_last_6mths', 'installment', 'int_rate', 'loan_amnt', 'open_acc', 'pub_rec', 'revol_util', 'sub_grade', 'term_months', 'total_acc']

Detected binary/OHE features (count=86). Showing first 60:
['addr_state_al', 'addr_state_ar', 'addr_state_az', 'addr_state_ca', 'addr_state_co', 'addr_state_ct', 'addr_state_fl', 'addr_state_ga', 'addr_state_il', 'addr_state_in', 'addr_state_ks', 'addr_state_ky', 'addr_state_la', 'addr_state_ma', 'addr_state_md', 'addr_state_mi', 'addr_state_mn', 'addr_state_mo', 'addr_state_ms', 'addr_state_nc', 'addr_state_nj', 'addr_state_nm', 'addr_state_nv', 'addr_state_ny', 'addr_state_oh', 'addr_state_ok', 'addr_state_or', 'addr_state_other', 'addr_state_pa', 'addr_state_sc', 'addr_state_tn', 'addr_state_tx', 'addr_state_ut', 'addr_state_va', 'a

In [ ]:
### Phase 2 — Pipeline Training
RobustScaler + Logistic Regression pipeline fitted on training set only. 
Scaler medians and IQRs computed exclusively from train and applied identically 
to val/test — no leakage. Pre-calibration metrics establish the baseline before 
calibration is addressed in the next phase.

In [3]:
# Phase 1 Step-2 (final): Fit RobustScaler + LogisticRegression(class_weight='balanced'),
# evaluate on train/val/test, save pipeline/scaler/metrics, print concise outputs.
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, confusion_matrix, precision_score, recall_score, f1_score

# ---------- CONFIG ----------
ARTIFACT_DIR = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts"
NUM_JSON = os.path.join(ARTIFACT_DIR, "numeric_features.json")
BIN_JSON = os.path.join(ARTIFACT_DIR, "binary_ohe_features.json")
CANONICAL_JSON = os.path.join(ARTIFACT_DIR, "canonical_feature_list.json")
PIPELINE_JOBLIB = os.path.join(ARTIFACT_DIR, "pipeline_robust_lr_v1.joblib")
SCALER_JOBLIB = os.path.join(ARTIFACT_DIR, "scaler_robust_v1.joblib")
METRICS_JSON = os.path.join(ARTIFACT_DIR, "metrics_robust_lr_v1.json")
CALIBRATION_CSV = os.path.join(ARTIFACT_DIR, "calibration_table_val.csv")
LIFT_CSV = os.path.join(ARTIFACT_DIR, "decile_lift_val.csv")
GUIDE_PATH = "/mnt/data/Credit_Risk_Project_Guide.pdf"
TARGET = "default_flag"
THRESHOLDS = [0.5, 0.25, 0.10]
N_BINS = 10
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------- SANITY: ensure splits are loaded ----------
if "df_train" not in globals() or "df_val" not in globals() or "df_test" not in globals():
    raise RuntimeError("Please load df_train, df_val, df_test into memory before running this cell.")

df_train = globals()["df_train"].reset_index(drop=True)
df_val   = globals()["df_val"].reset_index(drop=True)
df_test  = globals()["df_test"].reset_index(drop=True)

print(f"Loaded splits: train={df_train.shape}, val={df_val.shape}, test={df_test.shape}")

# ---------- 1) Load canonical & feature lists ----------
# Load canonical (if available)
ordered_cols = None
if os.path.exists(CANONICAL_JSON):
    with open(CANONICAL_JSON, "r") as f:
        canonical_obj = json.load(f)
    # try common keys
    ordered_cols = canonical_obj.get("ordered_columns_with_target_last") or canonical_obj.get("ordered_columns_with_target") \
                   or canonical_obj.get("ordered_columns") or canonical_obj.get("ordered_cols") or None
    canonical_features = canonical_obj.get("features") or canonical_obj.get("feature_list") or canonical_obj.get("features", None)
    # if ordered_cols not available, but a features list is present use it
    if ordered_cols is None and canonical_features:
        ordered_cols = canonical_features
    print("Canonical ordering loaded from:", CANONICAL_JSON)
else:
    print("Canonical file not found at:", CANONICAL_JSON)

# Load numeric & binary lists (fallback to autodetect)
if os.path.exists(NUM_JSON):
    with open(NUM_JSON, "r") as f:
        numeric_features = json.load(f)
    print(f"Loaded numeric features from {NUM_JSON} (count={len(numeric_features)})")
else:
    numeric_features = [c for c in df_train.columns if str(df_train[c].dtype).startswith("float")]
    print("numeric_features.json not found; autodetected numeric features:", numeric_features)

if os.path.exists(BIN_JSON):
    with open(BIN_JSON, "r") as f:
        binary_ohe_features = json.load(f)
    print(f"Loaded binary/OHE features from {BIN_JSON} (count={len(binary_ohe_features)})")
else:
    # int8 columns except target
    binary_ohe_features = [c for c in df_train.columns if str(df_train[c].dtype).startswith("int8") and c != TARGET]
    print("binary_ohe_features.json not found; autodetected binary/OHE features (first 40):", binary_ohe_features[:40])

# Build final feature list and enforce canonical ordering if available
features = [c for c in (numeric_features + binary_ohe_features) if c != TARGET]
if ordered_cols:
    ordered_feats = [c for c in ordered_cols if c != TARGET and c in features]
    # append any features not found in canonical ordering
    remaining = [c for c in features if c not in ordered_feats]
    features = ordered_feats + remaining
    print(f"Using canonical-aligned feature order, total features = {len(features)}")
else:
    print(f"Using numeric+binary order, total features = {len(features)}")

# Safety: ensure all features exist in each df; if missing create zeros (and warn)
def ensure_features_exist(df, features):
    missing = [c for c in features if c not in df.columns]
    if missing:
        print(f"  [WARN] Missing {len(missing)} columns; creating zeros: {missing}")
        for c in missing:
            # guess dtype: float if in numeric_features else int8
            if c in numeric_features:
                df[c] = 0.0
            else:
                df[c] = 0
    # reorder columns (features + target last)
    # ensure target exists
    if TARGET not in df.columns:
        raise RuntimeError(f"Target column '{TARGET}' not found in dataframe.")
    return df[features + [TARGET]].copy()

df_train = ensure_features_exist(df_train, features)
df_val = ensure_features_exist(df_val, features)
df_test = ensure_features_exist(df_test, features)

# ---------- 2) Build preprocessor & pipeline ----------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numeric_features),
        ("bin", "passthrough", binary_ohe_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

clf = LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=2000, random_state=42)
pipeline = Pipeline([("preprocessor", preprocessor), ("clf", clf)])

# ---------- 3) Prepare X/y ----------
X_train = df_train[features].astype({c: "float32" for c in numeric_features})
y_train = df_train[TARGET].astype(int)
X_val   = df_val[features].astype({c: "float32" for c in numeric_features})
y_val   = df_val[TARGET].astype(int)
X_test  = df_test[features].astype({c: "float32" for c in numeric_features})
y_test  = df_test[TARGET].astype(int)

print("Prepared X/y. Numeric columns cast to float32 for memory stability.")

# ---------- 4) Fit pipeline ----------
print("\nFitting pipeline on TRAIN (this computes RobustScaler medians/IQR & LR coeffs)...")
pipeline.fit(X_train, y_train)
print("Fitting complete.")

# Save pipeline + scaler
joblib.dump(pipeline, PIPELINE_JOBLIB)
# extract fitted scaler for audit
fitted_num = pipeline.named_steps["preprocessor"].named_transformers_["num"]
joblib.dump(fitted_num, SCALER_JOBLIB)
print("Saved pipeline:", PIPELINE_JOBLIB)
print("Saved numeric scaler:", SCALER_JOBLIB)
print("Project guide (informational):", GUIDE_PATH)

# ---------- helpers for metrics ----------
def ks_stat(y_true, y_score):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    total_pos = int((dfc["y"] == 1).sum())
    total_neg = int((dfc["y"] == 0).sum())
    if total_pos == 0 or total_neg == 0:
        return float("nan")
    dfc["cum_pos"] = (dfc["y"] == 1).cumsum()
    dfc["cum_neg"] = (dfc["y"] == 0).cumsum()
    dfc["tpr"] = dfc["cum_pos"] / total_pos
    dfc["fpr"] = dfc["cum_neg"] / total_neg
    ks = (dfc["tpr"] - dfc["fpr"]).max()
    return float(ks)

def calibration_table(y_true, y_score, n_bins=10):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    n = len(dfc)
    size = max(1, n // n_bins)
    rows = []
    for i in range(n_bins):
        start = i * size
        end = (i+1) * size if i < n_bins-1 else n
        bucket = dfc.iloc[start:end]
        if bucket.shape[0] == 0:
            continue
        rows.append({
            "decile": i+1,
            "n_rows": int(bucket.shape[0]),
            "avg_pred_pd": float(bucket["score"].mean()),
            "actual_default_rate": float(bucket["y"].mean()),
            "defaults": int(bucket["y"].sum())
        })
    return pd.DataFrame(rows)

def decile_lift_table(y_true, y_score, n_bins=10):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    n = len(dfc)
    size = max(1, n // n_bins)
    total_defaults = int(dfc["y"].sum())
    rows = []
    for i in range(n_bins):
        start = i * size
        end = (i+1) * size if i < n_bins-1 else n
        bucket = dfc.iloc[start:end]
        defaults = int(bucket["y"].sum())
        rows.append({
            "decile": i+1,
            "n_rows": int(bucket.shape[0]),
            "defaults": defaults,
            "default_rate": float(defaults / bucket.shape[0]) if bucket.shape[0] > 0 else 0.0,
            "share_of_total_defaults": float(defaults / total_defaults) if total_defaults > 0 else 0.0
        })
    return pd.DataFrame(rows)

# ---------- 5) Score & print concise metrics ----------
results = {}
for split_name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print("\n" + "-"*56)
    print(f"SCORING -> {split_name.upper()} (rows={len(X)})")
    y_proba = pipeline.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, y_proba)
    ks = ks_stat(y, y_proba)
    brier = brier_score_loss(y, y_proba)
    print(f"  AUC: {auc:.5f} | KS: {ks:.5f} | Brier: {brier:.6f}")

    # threshold metrics
    for t in THRESHOLDS:
        y_pred = (y_proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0,1]).ravel()
        prec = precision_score(y, y_pred, zero_division=0)
        rec = recall_score(y, y_pred, zero_division=0)
        f1 = f1_score(y, y_pred, zero_division=0)
        print(f"   thr={t:.2f} -> TP={tp}, FP={fp}, FN={fn}, TN={tn} | prec={prec:.3f}, rec={rec:.3f}, f1={f1:.3f}")

    # calibration + lift (save for validation)
    cal_tab = calibration_table(y, y_proba, n_bins=N_BINS)
    lift_tab = decile_lift_table(y, y_proba, n_bins=N_BINS)
    print("\n  Calibration deciles (decile | avg_pred_pd | actual_rate | rows | defaults):")
    print(cal_tab.to_string(index=False, float_format='{:0.6f}'.format))
    print("\n  Decile lift (decile | default_rate | defaults | share_of_total_defaults):")
    print(lift_tab.to_string(index=False, float_format='{:0.6f}'.format))

    if split_name == "val":
        cal_tab.to_csv(CALIBRATION_CSV, index=False)
        lift_tab.to_csv(LIFT_CSV, index=False)
        print(f"\n  [SAVED] validation calibration -> {CALIBRATION_CSV}")
        print(f"  [SAVED] validation lift -> {LIFT_CSV}")

    results[split_name] = {
        "auc": float(auc),
        "ks": float(ks),
        "brier": float(brier),
        "n_rows": int(len(X))
    }

# ---------- 6) Save metrics ----------
with open(METRICS_JSON, "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved metrics JSON ->", METRICS_JSON)
print("\nPipeline training & evaluation complete. Artifacts saved to:", ARTIFACT_DIR)


Loaded splits: train=(941745, 105), val=(201802, 105), test=(201803, 105)
Canonical ordering loaded from: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/canonical_feature_list.json
Loaded numeric features from /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/numeric_features.json (count=18)
Loaded binary/OHE features from /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/binary_ohe_features.json (count=86)
Using canonical-aligned feature order, total features = 104
Prepared X/y. Numeric columns cast to float32 for memory stability.

Fitting pipeline on TRAIN (this computes RobustScaler medians/IQR & LR coeffs)...
Fitting complete.
Saved pipeline: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/pipeline_robust_lr_v1.joblib
Saved numeric scaler: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/scaler_robust_v1.joblib
Project guide (informational): /mnt/data/Credit_Risk_Project_Guide.pdf


In [ ]:
### Phase 3 — Platt Calibration

**Problem:** Pre-calibration Brier score of 0.2154 shows the raw LR probabilities 
were poorly calibrated — predicted PDs did not match observed default rates.

**Why this matters for credit risk:** discrimination (AUC/KS) and calibration are 
separate properties. A model can rank borrowers correctly while still producing 
probabilities that are wrong in absolute terms — which directly distorts ECL 
estimates and risk-based pricing.

**Method:** Platt calibration (sigmoid) using `cv='prefit'` — base model trained on 
train, calibrator fitted on validation (val unseen during training, so no leakage), 
test set held out entirely until final evaluation.

**Result:** Brier dropped from 0.2154 → 0.1446 on validation. AUC and KS unchanged 
— calibration corrects probability scale without affecting rank ordering, exactly 
as expected.

*Note: `cv='prefit'` raises a FutureWarning in sklearn 1.6+. Equivalent modern 
syntax uses `FrozenEstimator`. Behavior is identical; retained as-is.*

In [3]:
# === Reload splits + Run Platt calibration (timestamp fix + full flow) ===
import os
import json
import joblib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# ---------- CONFIG (update paths if yours differ) ----------
ARTIFACT_DIR = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts"
TRAIN_PATH = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_train_labeled.parquet"
VAL_PATH   = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_val_labeled.parquet"
TEST_PATH  = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_test_labeled.parquet"
BASE_PIPELINE_FN = "pipeline_robust_lr_v1.joblib"
BASE_METRICS_FN = "metrics_robust_lr_v1.json"
GUIDE_PATH = "/mnt/data/Credit_Risk_Project_Guide.pdf"  # informational
TARGET = "default_flag"
N_BINS = 10

os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ---------- timestamp (timezone-aware to avoid deprecation) ----------
ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

CALIBRATED_PIPELINE_FN = f"pipeline_robust_lr_v1_calibrated_platt_{ts}.joblib"
METRICS_POST_FN = f"metrics_robust_lr_v1_post_platt_{ts}.json"
CALIB_CSV_FN = f"calibration_table_val_post_platt_{ts}.csv"
LIFT_CSV_FN = f"decile_lift_val_post_platt_{ts}.csv"
README_FN = f"artifacts_readme_post_platt_{ts}.txt"

CALIBRATED_PIPELINE_PATH = os.path.join(ARTIFACT_DIR, CALIBRATED_PIPELINE_FN)
METRICS_POST_PATH = os.path.join(ARTIFACT_DIR, METRICS_POST_FN)
CALIB_CSV_PATH = os.path.join(ARTIFACT_DIR, CALIB_CSV_FN)
LIFT_CSV_PATH = os.path.join(ARTIFACT_DIR, LIFT_CSV_FN)
README_PATH = os.path.join(ARTIFACT_DIR, README_FN)

# ---------- helper functions ----------
def ks_stat(y_true, y_score):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    total_pos = int((dfc["y"] == 1).sum())
    total_neg = int((dfc["y"] == 0).sum())
    if total_pos == 0 or total_neg == 0:
        return float("nan")
    dfc["cum_pos"] = (dfc["y"] == 1).cumsum()
    dfc["cum_neg"] = (dfc["y"] == 0).cumsum()
    dfc["tpr"] = dfc["cum_pos"] / total_pos
    dfc["fpr"] = dfc["cum_neg"] / total_neg
    return float((dfc["tpr"] - dfc["fpr"]).max())

def calibration_deciles(y_true, y_score, n_bins=10):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    n = len(dfc)
    size = max(1, n // n_bins)
    rows = []
    for i in range(n_bins):
        start = i * size
        end = (i+1) * size if i < n_bins-1 else n
        bucket = dfc.iloc[start:end]
        if bucket.shape[0] == 0:
            continue
        rows.append({
            "decile": i+1,
            "n_rows": int(bucket.shape[0]),
            "avg_pred_pd": float(bucket["score"].mean()),
            "actual_default_rate": float(bucket["y"].mean()),
            "defaults": int(bucket["y"].sum())
        })
    return pd.DataFrame(rows)

def decile_lift_table(y_true, y_score, n_bins=10):
    dfc = pd.DataFrame({"y": y_true, "score": y_score})
    dfc = dfc.sort_values("score", ascending=False).reset_index(drop=True)
    n = len(dfc)
    size = max(1, n // n_bins)
    total_defaults = int(dfc["y"].sum())
    rows = []
    for i in range(n_bins):
        start = i * size
        end = (i+1) * size if i < n_bins-1 else n
        bucket = dfc.iloc[start:end]
        defaults = int(bucket["y"].sum())
        rows.append({
            "decile": i+1,
            "n_rows": int(bucket.shape[0]),
            "defaults": defaults,
            "default_rate": float(defaults / bucket.shape[0]) if bucket.shape[0] > 0 else 0.0,
            "share_of_total_defaults": float(defaults / total_defaults) if total_defaults > 0 else 0.0
        })
    return pd.DataFrame(rows)

# ---------- 0) Load data splits into memory (reset index) ----------
print("Loading data splits...")
df_train = pd.read_parquet(TRAIN_PATH).reset_index(drop=True)
df_val   = pd.read_parquet(VAL_PATH).reset_index(drop=True)
df_test  = pd.read_parquet(TEST_PATH).reset_index(drop=True)
print(f"Loaded: train {df_train.shape}, val {df_val.shape}, test {df_test.shape}")

# ---------- 1) Load or locate baseline pipeline ----------
pipeline_obj = globals().get("pipeline", None)
if pipeline_obj is None:
    pipeline_path = os.path.join(ARTIFACT_DIR, BASE_PIPELINE_FN)
    if not os.path.exists(pipeline_path):
        raise RuntimeError(f"No in-memory pipeline and baseline pipeline not found at: {pipeline_path}\nPlease fit & save pipeline first.")
    print("Loading baseline pipeline from:", pipeline_path)
    pipeline_obj = joblib.load(pipeline_path)
else:
    print("Using in-memory pipeline object.")

# ---------- 2) derive feature list (use global if available) ----------
if "features" in globals():
    features = globals()["features"]
else:
    # fallback: all columns except target
    features = [c for c in df_train.columns if c != TARGET]
print("Feature count:", len(features))

X_train = df_train[features].copy()
y_train = df_train[TARGET].astype(int).copy()
X_val   = df_val[features].copy()
y_val   = df_val[TARGET].astype(int).copy()
X_test  = df_test[features].copy()
y_test  = df_test[TARGET].astype(int).copy()

# ---------- 3) baseline metrics (pre-calibration) ----------
print("\n--- Baseline (pre-calibration) metrics ---")
baseline = {}
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    y_proba = pipeline_obj.predict_proba(X)[:,1]
    auc = roc_auc_score(y, y_proba)
    ks = ks_stat(y, y_proba)
    brier = brier_score_loss(y, y_proba)
    baseline[name] = {"auc": float(auc), "ks": float(ks), "brier": float(brier)}
    print(f" - {name}: AUC={auc:.5f} | KS={ks:.5f} | Brier={brier:.6f}")

# ---------- 4) Fit Platt calibrator on validation only (cv='prefit') ----------
print("\nFitting Platt (sigmoid) calibrator on validation (cv='prefit') ...")
calibrator = CalibratedClassifierCV(estimator=pipeline_obj, method="sigmoid", cv="prefit")

calibrator.fit(X_val, y_val)
print("Calibrator fitted on validation.")

# ---------- 5) Evaluate post-calibration and save CSVs ----------
print("\n--- Post-calibration metrics (Platt) ---")
post_metrics = {}
for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    y_proba_cal = calibrator.predict_proba(X)[:,1]
    auc = roc_auc_score(y, y_proba_cal)
    ks = ks_stat(y, y_proba_cal)
    brier = brier_score_loss(y, y_proba_cal)
    post_metrics[name] = {"auc": float(auc), "ks": float(ks), "brier": float(brier)}
    print(f" - {name}: AUC={auc:.5f} | KS={ks:.5f} | Brier={brier:.6f}")
    if name == "val":
        dec_tab = calibration_deciles(y, y_proba_cal, n_bins=N_BINS)
        lift_tab = decile_lift_table(y, y_proba_cal, n_bins=N_BINS)
        print("\nValidation calibration deciles (post-Platt):")
        print(dec_tab.to_string(index=False, float_format='{:0.6f}'.format))
        dec_tab.to_csv(CALIB_CSV_PATH, index=False)
        lift_tab.to_csv(LIFT_CSV_PATH, index=False)
        print(f"\nSaved: {CALIB_CSV_PATH}")
        print(f"Saved: {LIFT_CSV_PATH}")

# ---------- 6) Save calibrated classifier and metrics JSON ----------
joblib.dump(calibrator, CALIBRATED_PIPELINE_PATH)
print("\nSaved calibrated pipeline ->", CALIBRATED_PIPELINE_PATH)

metrics_out = {
    "timestamp": ts,
    "baseline": baseline,
    "post_platt": post_metrics,
    "notes": "Platt calibrator fitted on validation only. Baseline pipeline saved separately."
}
with open(METRICS_POST_PATH, "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Saved metrics JSON ->", METRICS_POST_PATH)

# ---------- 7) Save short README for audit ----------
readme_lines = [
    f"Artifacts produced at {datetime.now(timezone.utc).isoformat()} UTC",
    "",
    "Original (pre-calibration) artifacts preserved in this folder:",
    f" - pipeline baseline (kept): {BASE_PIPELINE_FN}",
    f" - metrics baseline (kept): {BASE_METRICS_FN}",
    f" - project guide (informational): {GUIDE_PATH}",
    "",
    "New artifacts produced by this run (Platt / sigmoid calibration):",
    f" - calibrated pipeline (platt sigmoidal): {CALIBRATED_PIPELINE_FN}",
    f" - post-calibration metrics (json): {os.path.basename(METRICS_POST_PATH)}",
    f" - validation calibration table (post-platt): {os.path.basename(CALIB_CSV_PATH)}",
    f" - validation decile lift (post-platt): {os.path.basename(LIFT_CSV_PATH)}",
    "",
    "Notes:",
    " - Calibrator was FIT on validation only (no leakage).",
    " - The calibrated pipeline should be used for scoring/production (it contains pipeline + platt mapping).",
    " - Original uncalibrated pipeline & metrics are preserved for audit."
]
with open(README_PATH, "w") as f:
    f.write("\n".join(readme_lines))
print("Saved README ->", README_PATH)

# leave calibrated object accessible
calibrated_clf = calibrator

print("\nPLATT calibration run complete.")


Loading data splits...
Loaded: train (941745, 105), val (201802, 105), test (201803, 105)
Loading baseline pipeline from: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/pipeline_robust_lr_v1.joblib
Feature count: 104

--- Baseline (pre-calibration) metrics ---
 - train: AUC=0.71378 | KS=0.31022 | Brier=0.215855
 - val: AUC=0.71399 | KS=0.31142 | Brier=0.215401
 - test: AUC=0.71397 | KS=0.31203 | Brier=0.216166

Fitting Platt (sigmoid) calibrator on validation (cv='prefit') ...
Calibrator fitted on validation.

--- Post-calibration metrics (Platt) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


 - train: AUC=0.71378 | KS=0.31022 | Brier=0.144650
 - val: AUC=0.71399 | KS=0.31142 | Brier=0.144630

Validation calibration deciles (post-Platt):
 decile  n_rows  avg_pred_pd  actual_default_rate  defaults
      1   20180     0.474212             0.455302      9188
      2   20180     0.334742             0.337364      6808
      3   20180     0.265077             0.270268      5454
      4   20180     0.218671             0.230278      4647
      5   20180     0.183750             0.194252      3920
      6   20180     0.154772             0.160902      3247
      7   20180     0.129175             0.131962      2663
      8   20180     0.105227             0.101437      2047
      9   20180     0.080576             0.075372      1521
     10   20182     0.050325             0.039392       795

Saved: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/calibration_table_val_post_platt_20251125T164455Z.csv
Saved: /Users/abhinavsaxena/Documents/Project/1/clean_data/sp

In [ ]:
### Phase 4 — Coefficient Table, Business Sense Check, and Final Test Metric

The coefficient table is the most important validation for a logistic regression 
PD model — more important than the AUC. It proves the model learned actual credit 
risk logic rather than spurious correlations.

Each coefficient is converted to an odds ratio (e^coefficient) and a percentage 
change in odds ((odds_ratio − 1) × 100). Because features were scaled with 
RobustScaler, each coefficient reflects the effect of a one-IQR change in that 
feature, not one raw unit.

Core risk drivers are checked explicitly against expected signs:
- `fico_mean` → expected negative (higher score, lower risk)
- `dti` → expected positive (higher debt burden, higher risk)
- `annual_inc` → expected negative (higher income, lower risk)
- `inq_last_6mths`, `delinq_2yrs`, `revol_util` → expected positive
- `credit_history_years` → expected negative

If any of these contradict expectation, it signals a conceptual problem 
regardless of AUC — and must be investigated before the model is trusted.

Finally, the test set Brier score is computed using the calibrated pipeline — 
the single unbiased measure of how well-calibrated this model is on data it 
has never influenced in any way (not training, not calibration fitting).

In [ ]:
**Note on revol_util coefficient direction:**
revol_util shows a negative coefficient (decreasing default risk), which is 
counter to typical credit risk intuition — higher revolving utilization usually 
signals financial stress. This is likely attributable to multicollinearity with 
correlated features (dti, revol_util_was_extreme). This is a known limitation 
of unregularized coefficient interpretation in the presence of correlated 
features and would be investigated via VIF analysis in a production model. 
It does not affect overall model discrimination (AUC) or calibration (Brier), 
which are validated independently. Flagged for remediation in next model version.

**Note on state-level coefficients:**
Several addr_state dummy variables appear among the largest coefficients 
(e.g., addr_state_ms, addr_state_or). These are not interpreted as strong 
standalone risk drivers — state-level effects in consumer lending are typically 
weak and can reflect sample-size artifacts in lower-frequency states rather 
than genuine geographic risk differences. Core underwriting variables 
(fico_mean, dti, annual_inc, credit_history_years) remain the primary 
interpretable risk drivers and behave as expected.

**Note on emp_length_yrs_was_missing:**
The flag indicating missing employment length is the single strongest predictor 
of increased default risk in the model (odds ratio 1.64). This suggests that 
borrowers who did not report employment length carry meaningfully higher credit 
risk than the imputed value (6.0 years) would suggest — possibly because 
non-disclosure correlates with unstable or informal employment. This validates 
the decision to retain missingness as an explicit flag rather than relying on 
median imputation alone.

In [1]:
# === Final Cell: Coefficient Table, Odds Ratios, and Test Brier Score ===
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss

# ---------- CONFIG ----------
ARTIFACT_DIR = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts"
TEST_PATH = "/Users/abhinavsaxena/Documents/Project/1/clean_data/splits/ready_test_labeled.parquet"
BASE_PIPELINE_PATH = os.path.join(ARTIFACT_DIR, "pipeline_robust_lr_v1.joblib")
NUM_JSON = os.path.join(ARTIFACT_DIR, "numeric_features.json")
BIN_JSON = os.path.join(ARTIFACT_DIR, "binary_ohe_features.json")
TARGET = "default_flag"

# Use the most recent calibrated pipeline saved by Cell 4
calibrated_files = sorted([f for f in os.listdir(ARTIFACT_DIR) 
                            if f.startswith("pipeline_robust_lr_v1_calibrated_platt_")])
CALIBRATED_PIPELINE_PATH = os.path.join(ARTIFACT_DIR, calibrated_files[-1])

COEF_CSV_PATH = os.path.join(ARTIFACT_DIR, "coefficients_lr_v1.csv")

# ---------- 1) Load feature lists ----------
with open(NUM_JSON, "r") as f:
    numeric_features = json.load(f)
with open(BIN_JSON, "r") as f:
    binary_ohe_features = json.load(f)

feature_names = numeric_features + binary_ohe_features
print(f"Total features: {len(feature_names)}")

# ---------- 2) Load base pipeline and extract coefficients ----------
base_pipeline = joblib.load(BASE_PIPELINE_PATH)
lr_model = base_pipeline.named_steps['clf']

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr_model.coef_[0],
    'odds_ratio': np.exp(lr_model.coef_[0])
}).sort_values('coefficient', ascending=False).reset_index(drop=True)

coef_df['pct_change_in_odds'] = (coef_df['odds_ratio'] - 1) * 100
coef_df['direction'] = coef_df['coefficient'].apply(
    lambda x: 'increases_default_risk' if x > 0 else 'decreases_default_risk'
)

coef_df.to_csv(COEF_CSV_PATH, index=False)
print(f"\nSaved coefficient table: {COEF_CSV_PATH}")

print("\nTop 10 — strongest INCREASE in default risk:")
print(coef_df.head(10).to_string(index=False))

print("\nTop 10 — strongest DECREASE in default risk:")
print(coef_df.tail(10).to_string(index=False))

# ---------- 3) Business sense check on core risk drivers ----------
core_drivers = ['fico_mean', 'dti', 'annual_inc', 'inq_last_6mths', 
                 'delinq_2yrs', 'revol_util', 'credit_history_years']
print("\nCore risk driver check:")
print(coef_df[coef_df['feature'].isin(core_drivers)]
      [['feature', 'coefficient', 'odds_ratio', 'direction']].to_string(index=False))

# ---------- 4) Test set Brier score (final calibrated model) ----------
df_test = pd.read_parquet(TEST_PATH).reset_index(drop=True)
X_test = df_test[feature_names]
y_test = df_test[TARGET]

calibrated_pipeline = joblib.load(CALIBRATED_PIPELINE_PATH)
test_pred = calibrated_pipeline.predict_proba(X_test)[:, 1]
test_brier = brier_score_loss(y_test, test_pred)

print(f"\nFinal Test Set Brier Score (post-Platt calibration): {test_brier:.6f}")

Total features: 104

Saved coefficient table: /Users/abhinavsaxena/Documents/Project/1/clean_data/splits/artifacts/coefficients_lr_v1.csv

Top 10 — strongest INCREASE in default risk:
                   feature  coefficient  odds_ratio  pct_change_in_odds              direction
emp_length_yrs_was_missing     0.497192    1.644098           64.409763 increases_default_risk
    purpose_small_business     0.376422    1.457063           45.706256 increases_default_risk
             addr_state_ms     0.354790    1.425881           42.588122 increases_default_risk
                     grade     0.257644    1.293878           29.387760 increases_default_risk
             addr_state_ar     0.254158    1.289375           28.937536 increases_default_risk
                 sub_grade     0.246681    1.279771           27.977074 increases_default_risk
                       dti     0.236144    1.266357           26.635663 increases_default_risk
             addr_state_ok     0.226444    1.254133     

In [ ]:
### Final Results — Logistic Regression Champion (Post-Calibration)

| Metric | Train | Validation | Test |
|---|---|---|---|
| AUC | 0.7138 | 0.7140 | 0.7140 |
| KS | 0.3102 | 0.3114 | 0.3120 |
| Brier (pre-calibration) | 0.2159 | 0.2154 | 0.2162 |
| Brier (post-calibration) | 0.1447 | 0.1446 | 0.144680 |

AUC of 0.714 is realistic for unsecured consumer lending — not inflated. 
Near-identical train/validation AUC confirms no overfitting. The calibration 
decile table shows predicted PD tracking actual default rate closely across 
all 10 risk bands, with no systematic over- or under-estimation in any segment.

---

### Known Limitations

- **VIF not computed** — L2 regularization provides implicit shrinkage of 
  correlated coefficients, partially mitigating this risk. Flagged for next version.
- **Calibration set overlap** — Platt calibrator fitted on validation, which was 
  not used in base model training (`cv='prefit'` protocol). Test set is the 
  unbiased final estimate.
- **No hyperparameter tuning** — C=1.0 used as standard baseline to keep the 
  model simple and directly comparable to challenger models.

---

### Artifacts Saved

| File | Contents |
|---|---|
| `pipeline_robust_lr_v1.joblib` | Base pipeline (pre-calibration) |
| `scaler_robust_v1.joblib` | RobustScaler — median/IQR for 18 numeric features |
| `metrics_robust_lr_v1.json` | Pre-calibration metrics |
| `pipeline_robust_lr_v1_calibrated_platt_[ts].joblib` | Final calibrated model — used for scoring |
| `metrics_robust_lr_v1_post_platt_[ts].json` | Post-calibration metrics |
| `calibration_table_val_post_platt_[ts].csv` | Validation calibration deciles |
| `decile_lift_val_post_platt_[ts].csv` | Validation lift table |
| `coefficients_lr_v1.csv` | Coefficient table, odds ratios, business sense check |

This logistic regression model is the interpretable baseline champion. 
XGBoost V1 (unconstrained) and V2 (monotonic constrained) are benchmarked 
against these metrics in the following notebooks.